# Imports

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.integrate import solve_ivp
from scipy.optimize import brentq

sigma = 5.670e-8                     # Stefan-Boltzmannova konstanta [W/(m^2 K^4)]
S0 = 1361.0                          # solarna konstanta [W/m^2]
leto = 3.156e7                       # [s]
C = 1000 * 4186 * 50                 # toplotna kapaciteta 50 m oceana na m^2 [J/(m^2 K)]

# Model 0D
Q0 = S0 / 4                          # povprečna vpadna gostota toka [W/m^2]

# Model 1D (North, Cahalan, Coakley 1981)
Q1, A, B, D = 334.4, 203.3, 2.09, 0.649
a0, a2, b0, s2, T_led = 0.681, -0.202, 0.38, -0.482, -10.0
N = 60
dx = 1.0 / N
x = (np.arange(N) + 0.5) * dx        # središča celic, x = sin(phi)
xr = np.arange(N + 1) * dx           # robovi celic
P2 = (3 * x**2 - 1) / 2

# Funkcije

In [ ]:
# --- Model 0D ---
def T_ravn(alfa, eps, mu=1.0):
    """Ravnovesna temperatura [K] pri stalnem albedu: mu Q0 (1 - alfa) = eps sigma T^4."""
    # TODO
    pass


def alfa_led(T, a_led=0.62, a_voda=0.30, T1=250.0, T2=280.0):
    """Albedo z ledeno povratno zanko: a_led pod T1, a_voda nad T2, vmes linearno."""
    return np.clip(a_led + (a_voda - a_led) * (T - T1) / (T2 - T1), min(a_led, a_voda), max(a_led, a_voda))


def neto_0d(T, mu=1.0, eps=0.61, alfa=alfa_led):
    """Neto gostota toka [W/m^2]: absorbirano sončno minus izsevano. alfa je funkcija T."""
    # TODO: mu Q0 (1 - alfa(T)) - eps sigma T^4
    pass


def potek_0d(T0, t_kon, **kw):
    """Časovni potek C dT/dt = neto_0d(T) s solve_ivp. Vrne (t [leta], T [K])."""
    t_eval = np.linspace(0, t_kon, 400)
    # TODO: solve_ivp z desno stranjo neto_0d(T, **kw) / C
    return t_eval / leto, np.full_like(t_eval, T0)


def ravnovesja_0d(mu=1.0, eps=0.61, Tmin=150.0, Tmax=350.0):
    """Vse ničle neto_0d na [Tmin, Tmax] in njihova stabilnost (stabilno, če d(neto)/dT < 0)."""
    T = np.linspace(Tmin, Tmax, 2001)
    f = neto_0d(T, mu, eps)
    out = []
    # TODO: poišči spremembe predznaka, ničlo z brentq, stabilnost iz numeričnega odvoda
    pass
    return out


# --- Model 1D ---
def koalbedo(T):
    """Delež absorbirane svetlobe: a0 + a2 P2(x) brez ledu, b0 na ledu (T < T_led)."""
    return np.where(T < T_led, b0, a0 + a2 * P2)


def korak_1d(T, mu, dt, D=D, A=A):
    """En korak FTCS za C dT/dt = mu Q1 s(x) a(x,T) - (A + B T) + D d/dx[(1 - x^2) dT/dx].
    Na robovih (ekvator x = 0, pol x = 1) je tok nič."""
    s = 1 + s2 * P2
    # TODO: tok na notranjih robovih celic (1 - xr^2) (T_{i+1} - T_i)/dx, divergenca, nova T
    pass


def ravnovesje_1d(T, mu=1.0, D=D, A=A, leta=15):
    """Iz začetnega profila T [°C] integrira leta let (dt iz pogoja stabilnosti) in vrne končni profil."""
    dt = min(0.4 * C * dx**2 / D, 5 * 86400) if D > 0 else 5 * 86400
    for _ in range(int(leta * leto / dt)):
        T = korak_1d(T, mu, dt, D, A)
    return T


def ledena_meja(T):
    """Geografska širina [°] ledene meje (0 = vse pod ledom, 90 = brez ledu)."""
    led = T < T_led
    if led.all():
        return 0.0
    if not led.any():
        return 90.0
    return np.degrees(np.arcsin(xr[np.argmax(led)]))

# Testno območje

In [ ]:
# P1
print(f"P1: T_e (alfa = 0.3, eps = 1) = {T_ravn(0.3, 1.0):.1f} K, eps(288 K) = {Q0 * 0.7 / (sigma * 288**4):.3f}")
# P2
eps0 = Q0 * 0.7 / (sigma * 288**4)
lam0 = 4 * eps0 * sigma * 288**3
print(f"P2: 4 eps sigma T^3 = {lam0:.2f} W/(m^2 K), tau = {C / lam0 / leto:.2f} let, dT(2xCO2) = {3.7 / lam0:.2f} K")
# P4
sa = a0 + s2 * a2 / 5                              # <P2^2> = 1/5
print(f"P4: <s a> = {sa:.4f}, T brez ledu = {(Q1 * sa - A) / B:.2f} °C, "
      f"T snežne kepe (mu = 1) = {(Q1 * b0 - A) / B:.2f} °C")
mu_t = (T_led + A / B) / (Q1 * b0 * (1 / B - s2 / 2 / (B + 6 * D)))
print(f"P4: mu odtalitve snežne kepe = {mu_t:.3f}")

# Naloga 1 - Slika 1

In [ ]:
fig = plt.figure(figsize=(12, 4))
ax = fig.add_subplot(121)
konst = lambda T: 0.3 + 0 * T                      # stalen albedo
# TODO: T_ravn in tau za eps = 1 in 0.61; potek iz T0 = 200, 288, 330 K; 2xCO2 (Preglednica 1)
pass
ax.set_xlabel("$t$ [leta]")
ax.set_ylabel("$T$ [K]")
ax.set_title(r"stalen albedo $\alpha = 0{,}3$", loc="left")
ax.legend()
ax = fig.add_subplot(122)
Ts = np.linspace(200, 330, 300)
ax.plot(Ts, Q0 * (1 - 0.3) + 0 * Ts, label=r"absorbirano, $\alpha = 0{,}3$")
ax.plot(Ts, eps0 * sigma * Ts**4, label=rf"izsevano, $\varepsilon = {eps0:.2f}$")
ax.plot(Ts, sigma * Ts**4, "--", label=r"izsevano, $\varepsilon = 1$")
ax.set_xlabel("$T$ [K]")
ax.set_ylabel("gostota toka [W/m$^2$]")
ax.legend()
fig.tight_layout()
plt.show()

# Naloga 2 - Slika 2

In [ ]:
fig = plt.figure(figsize=(12, 4))
ax = fig.add_subplot(121)
Ts = np.linspace(200, 330, 500)
for mu in (0.9, 1.0, 1.1):
    ax.plot(Ts, mu * Q0 * (1 - alfa_led(Ts)), label=rf"absorbirano, $\mu = {mu}$")
ax.plot(Ts, eps0 * sigma * Ts**4, "k", label="izsevano")
# TODO: ravnovesja in stabilnost za mu = 0.9, 1.0, 1.1 (Preglednica 2); označi jih na grafu
pass
ax.set_xlabel("$T$ [K]")
ax.set_ylabel("gostota toka [W/m$^2$]")
ax.legend(fontsize=8)
ax = fig.add_subplot(122)
# TODO: potek pri mu = 1 iz T0 = 200 ... 320 K
pass
ax.set_xlabel("$t$ [leta]")
ax.set_ylabel("$T$ [K]")
ax.set_title(r"$\mu = 1$", loc="left")
fig.tight_layout()
plt.show()

# Naloga 3 - Slika 3

In [ ]:
T_zac = np.full(N, 15.0)
fi = np.degrees(np.arcsin(x))
primeri = {}
# TODO: ravnovesni profil za D, D = 0, 5 D in 2xCO2 (A - 3.7); srednja T, T na ekvatorju in polu, ledena meja (Preglednica 3)
pass
fig = plt.figure(figsize=(12, 4))
ax = fig.add_subplot(121)
for ime, T in primeri.items():
    ax.plot(fi, T, label=ime)
ax.axhline(T_led, color="k", ls=":")
ax.set_xlabel(r"geografska širina $\varphi$ [°]")
ax.set_ylabel("$T$ [°C]")
ax.legend()
ax = fig.add_subplot(122)
T = primeri["$D$"]
s = 1 + s2 * P2
ax.plot(fi, Q1 * s * koalbedo(T), label="absorbirano")
ax.plot(fi, A + B * T, label="izsevano")
ax.fill_between(fi, Q1 * s * koalbedo(T), A + B * T, color="0.85")
ax.set_xlabel(r"geografska širina $\varphi$ [°]")
ax.set_ylabel("gostota toka [W/m$^2$]")
ax.legend()
fig.tight_layout()
plt.show()

# Naloga 4 - Slika 4

In [ ]:
mus = np.round(np.arange(1.40, 0.80 - 1e-9, -0.02), 2)
# TODO: zmanjšuj mu od 1.40 do 0.80 in nato povečuj nazaj; začetni profil = prejšnje ravnovesje (Preglednica 4)
dol = gor = np.zeros((len(mus), 2))
fig = plt.figure(figsize=(12, 4))
ax = fig.add_subplot(121)
ax.plot(mus, dol[:, 0], "o-", ms=3, label="zmanjševanje $\\mu$")
ax.plot(mus, gor[:, 0], "s-", ms=3, label="povečevanje $\\mu$")
ax.set_xlabel(r"$\mu = S/S_0$")
ax.set_ylabel(r"$\langle T\rangle$ [°C]")
ax.legend()
ax = fig.add_subplot(122)
ax.plot(mus, dol[:, 1], "o-", ms=3, label="zmanjševanje $\\mu$")
ax.plot(mus, gor[:, 1], "s-", ms=3, label="povečevanje $\\mu$")
ax.set_xlabel(r"$\mu = S/S_0$")
ax.set_ylabel("ledena meja [°]")
ax.legend()
fig.tight_layout()
plt.show()